# TP4 — Réseau de neurones codé en numpy (eau)

[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/earthai-tech/lab-courses/blob/main/inphb/prepa/notebooks/PREPA_TP4_reseau_numpy.ipynb)
Objectif : propagation avant, rétropropagation, vérification du gradient, puis entraînement.

> Données : dossier `PREPA_TP_donnees`. Règle du module : **numpy uniquement**, pas de bibliothèque d’apprentissage.

In [ ]:
# Configuration : Colab ou Jupyter local. Exécutez cette cellule en premier.
import os, shutil, urllib.request
DONNEES = ['forages_socle.csv']
URL = 'https://raw.githubusercontent.com/earthai-tech/lab-courses/main/inphb/prepa/data/'
for f in DONNEES:
    if os.path.exists(f):
        continue
    if os.path.exists(os.path.join('..', 'data', f)):
        shutil.copy(os.path.join('..', 'data', f), f)   # kit hors ligne / dépôt cloné
    else:
        urllib.request.urlretrieve(URL + f, f)            # Colab : téléchargement
    print('OK', f)


In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
f = pd.read_csv('forages_socle.csv')
X = np.c_[(f.sfi - f.sfi.mean()) / f.sfi.std(),
          (f.epaisseur_alteration_m - f.epaisseur_alteration_m.mean()) / f.epaisseur_alteration_m.std()]
y = (f.debit_m3h > 1).astype(float).values[:, None]
rng = np.random.default_rng(0); h = 8
W1 = rng.normal(0, .8, (2, h)); b1 = np.zeros(h); W2 = rng.normal(0, .8, (h, 1)); b2 = np.zeros(1)

## 1. Propagation avant et perte

In [ ]:
def forward(X, W1, b1, W2, b2):
    A1 = np.tanh(X @ W1 + b1)
    P = 1 / (1 + np.exp(-(A1 @ W2 + b2)))
    return A1, P
def perte(P, y):
    return -np.mean(y * np.log(P + 1e-12) + (1 - y) * np.log(1 - P + 1e-12))
A1, P = forward(X, W1, b1, W2, b2); print('perte initiale :', perte(P, y))

## 2. Rétropropagation
À COMPLÉTER : dérivées de la perte par rapport à W2, b2, W1, b1.

In [ ]:
def backward(X, y, A1, P, W2):
    dZ2 = (P - y) / len(y)
    dW2, db2 = A1.T @ dZ2, dZ2.sum(0)
    dZ1 = (dZ2 @ W2.T) * (1 - A1**2)
    dW1, db1 = X.T @ dZ1, dZ1.sum(0)
    return dW1, db1, dW2, db2

## 3. Vérification numérique du gradient (indispensable)

In [ ]:
dW1, db1, dW2, db2 = backward(X, y, A1, P, W2)
eps = 1e-5; E = np.zeros_like(W1); E[0, 0] = eps
num = (perte(forward(X, W1 + E, b1, W2, b2)[1], y) - perte(forward(X, W1 - E, b1, W2, b2)[1], y)) / (2 * eps)
print('analytique :', dW1[0, 0], ' numérique :', num)

## 4. Entraînement

In [ ]:
pertes = []
for it in range(3000):
    A1, P = forward(X, W1, b1, W2, b2); pertes.append(perte(P, y))
    dW1, db1, dW2, db2 = backward(X, y, A1, P, W2)
    W1 -= dW1; b1 -= db1; W2 -= dW2; b2 -= db2
print('accuracy entraînement :', ((P > .5) == y).mean())
plt.plot(pertes); plt.xlabel('itération'); plt.ylabel('entropie croisée'); plt.show()

## Bonus — généralisation
Séparer 70 % / 30 %, refaire l’entraînement pour h = 1, 4, 16, 64 et comparer l’accuracy d’entraînement et de test. Conclusion ?